[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

# 用 RNN 预测发动机故障

在这个实操里，目标是预测发动机的故障。训练数据集由发动机上多个传感器直到故障为止的时间序列组成。测试数据集由这些时间序列的开头和故障日期组成。

我们将构建一个简单的 RNN，输入是刻画发动机的多维时间序列，学习它的参数以在每个时刻预测故障时间。一开始，没有任何输入数据时，最好的预测应该是数据集中故障时间的平均值；随着越来越多的数据喂给 RNN，它应该给出越来越好的估计。

数据集由 [NASA](https://ti.arc.nasa.gov/tech/dash/groups/pcoe/prognostic-data-repository/#turbofan) 提供，也可以看 [Kaggle](https://www.kaggle.com/datasets/suriyachayatummagoon/cmapssdata?select=Damage+Propagation+Modeling.pdf)


In [ ]:
import torch
import torch.nn as nn
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import gamma
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns

In [ ]:
torch.__version__

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print('Using gpu: %s ' % torch.cuda.is_available())

## 1. 下载数据

这一步只需要做一次！

你可以在 [NASA](https://ti.arc.nasa.gov/tech/dash/groups/pcoe/prognostic-data-repository/#turbofan) 的网站上找到数据，也可以在 [Kaggle](https://www.kaggle.com/datasets/suriyachayatummagoon/cmapssdata?select=Damage+Propagation+Modeling.pdf) 上，或者在我的网站上：


In [ ]:
%mkdir data
%cd data

In [ ]:
!wget 'https://www.di.ens.fr/~lelarge/CMAPSSData.zip'

In [ ]:
!unzip CMAPSSData.zip

In [ ]:
%cd ..

## 2. 加载数据


In [ ]:
def get_CMAPSSData(nb_file):
    # 从文件获取数据并预处理（归一化并转成 pandas）
    dataset_train = pd.read_csv('./data/train_FD00{}.txt'.format(nb_file),
                                sep=' ', header=None).drop([26, 27], axis=1)
    dataset_test = pd.read_csv('./data/test_FD00{}.txt'.format(nb_file),
                               sep=' ', header=None).drop([26, 27], axis=1)
    test_truth = pd.read_csv('./data/RUL_FD00{}.txt'.format(nb_file),
                             sep=' ', header=None).drop([1], axis=1)
    col_names = ['id', 'cycle', 'setting1', 'setting2', 'setting3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8',
                 's9',
                 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']
    dataset_train.columns = col_names
    dataset_test.columns = col_names
    test_truth.columns = ['more']
    test_truth['id'] = test_truth.index + 1
    rul = pd.DataFrame(dataset_test.groupby('id')['cycle'].max()).reset_index()
    rul.columns = ['id', 'max']
    test_truth['rtf'] = test_truth['more'] + rul['max']
    test_truth.drop('more', axis=1, inplace=True)
    dataset_test = dataset_test.merge(test_truth, on=['id'], how='left')
    dataset_test['ttf'] = dataset_test['rtf'] - dataset_test['cycle']
    dataset_test.drop('rtf', axis=1, inplace=True)
    dataset_train['ttf'] = dataset_train.groupby(['id'])['cycle'].transform(max) - dataset_train['cycle']
    features_col_name = ['setting1', 'setting2', 'setting3', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8',
                         's9', 's10', 's11',
                         's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']
    target_col_name = 'ttf'
    relevant_features_col_name = []
    for col in features_col_name:
        if not (len(dataset_train[col].unique()) == 1):
            relevant_features_col_name.append(col)
    sc = MinMaxScaler()
    dataset_train[features_col_name] = sc.fit_transform(dataset_train[features_col_name])
    dataset_test[features_col_name] = sc.transform(dataset_test[features_col_name])
    return dataset_train, dataset_test, relevant_features_col_name, target_col_name


def to_lists_of_tensors(dataset, features_col_name, target_col_name):
    # 把 pandas df 转成张量列表（供 pytorch 用）
    X, y = [], []
    nb_sequences = max(dataset['id'])
    for i in range(1, nb_sequences + 1):
        df_zeros = dataset.loc[dataset['id'] == i]
        df_one_x = df_zeros[features_col_name]
        df_one_y = df_zeros[target_col_name]
        X.append(torch.from_numpy(np.expand_dims(df_one_x.values, 1)).type(torch.FloatTensor))
        y.append(torch.from_numpy(df_one_y.values).type(torch.FloatTensor))
    return X, y


def convert_train_and_test_to_appropriate_format(dataset_train, dataset_test, features_col_name, target_col_name):
    # 取 2 个数据集（train 和 test）并转成张量列表
    X_train, y_train = to_lists_of_tensors(dataset_train, features_col_name, target_col_name)
    X_test, y_test = to_lists_of_tensors(dataset_test, features_col_name, target_col_name)
    return X_train, y_train, X_test, y_test


In [ ]:
%pycat ./data/readme.txt

In [ ]:
dataset_train, dataset_test, features_col_name, target_col_name = get_CMAPSSData(1)
X_train, y_train, X_test, y_test = convert_train_and_test_to_appropriate_format(dataset_train, dataset_test,
                                                                                    features_col_name, target_col_name)

In [ ]:
dataset_train.head()

这里我用 sklearn 的 [`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) 做了最少的数据预处理，把每个特征缩放到 (0,1)。

`X_train` 是一个列表，每个元素的形状是 (length_of_sequence,1,number_of_sensors)，其中第二个维度值为 1，对应 batch size。和课程里一样，我们不是按 batch 处理序列，而是一个一个地处理。


In [ ]:
X_train[0].shape

## 3. WTTE-RNN 模型

这里我们采用的方法灵感来自这篇[博客](https://ragulpr.github.io/2016/12/22/WTTE-RNN-Hackless-churn-modeling/#wtte-rnn-produces-risk-embeddings)。

你首先需要定义一个 GRU（或 LSTM），输入是形状为 (length_of_sequence,1,number_of_sensors) 的序列，输出是形状为 (length_of_sequence,2) 的序列（把 GRU 的输出过一个线性层得到）。因为我们想要正数，所以取指数。


In [ ]:
class GRUnet(nn.Module):
    def __init__(self, dim_input, num_layers, dim_hidden, dim_output=2):
        super(GRUnet, self).__init__()
        #
        # 你的代码
        #
        

    def forward(self, x):
        #
        # 你的代码
        #
        

测试你的网络


In [ ]:
model = GRUnet(dim_input=len(features_col_name), num_layers=3, dim_hidden=50)
model = model.to(device)

In [ ]:
output = model(X_train[0].to(device))

In [ ]:
output.shape

为了学习 RNN 的参数，你需要指定一个损失。这里我们采用可靠性理论里的一个标准做法：把故障时间建模为[威布尔（Weibull）随机变量](http://reliawiki.org/index.php/The_Weibull_Distribution)

$$
\mathbb{P}(X>t) = \exp\left( \frac{t}{\eta}\right)^{\beta},
$$
其中 $\eta$ 是尺度参数，$\beta$ 是形状参数。

注意威布尔分布的均值是：
$$
\mathbb{E}[X] = \eta \Gamma(1+1/\beta),
$$
其中 $\Gamma$ 是[伽马函数](https://en.wikipedia.org/wiki/Gamma_function)。

在我们的例子里，我们把 RNN 的 2 个输出解释为参数 $\eta$ 和 $\beta$ 的估计。为了设计损失，我们计算对数似然：
\begin{eqnarray*}
\log f(t) &=& \log\left( \frac{\beta}{\eta}\right) +(\beta -1)\log\left(\frac{t}{\eta}\right) -\left(\frac{t}
{\eta} \right)^{\beta}\\
&=& \log \beta +\beta \log\left(\frac{t}{\eta}\right) -\log t-\left(\frac{t}
{\eta} \right)^{\beta}
\end{eqnarray*}

定义一个对应负对数似然的损失函数（给 $t$ 加一个很小的参数 $\epsilon$，避免计算 $\log 0$）。


In [ ]:
class weibull_loss(nn.Module):
    def __init__(self):
        super(weibull_loss, self).__init__()
        self.epsilon = 1e-6

    def forward(self, output, y):
        #
        # 你的代码
        #

测试你的损失函数。


In [ ]:
loss_fn = weibull_loss()
loss_fn(output.squeeze(),y_train[0].to(device))

## 4. 训练你的模型

编写你的训练和测试循环。

你可能想用一个调度器，比如 `torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=1, verbose='True',threshold=0.001)`


In [ ]:
def train_epoch(X_train, y_train, model, loss_fn, optimizer, device):
    # 在整个训练数据集上训练模型一个 epoch
    # 返回这个 epoch 对应的损失
    #
    # 你的代码
    #


def test_epoch(X_test, y_test, model, loss_fn, device):
    # 在整个测试数据集上评估模型
    # 返回对应的损失
    #
    # 你的代码
    #


def fit(model, X_train, y_train, X_test, y_test, optimizer, loss_fn, nb_epochs, device):
    # 通过训练 nb_epochs 次来拟合模型
    # 你可能想用一个调度器
    #
    # 你的代码
    #

In [ ]:
model = GRUnet(dim_input=len(features_col_name), num_layers=3, dim_hidden=50,dim_output=2)
model = model.to(device)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = weibull_loss()
nb_epochs = 50

In [ ]:
model, train_loss_t, test_loss_t = fit(model, X_train, y_train, X_test, y_test, optimizer, loss_fn, nb_epochs,device)

In [ ]:
def plot_losses(train_loss_t, test_loss_t):
    nb_epochs = len(train_loss_t)
    plt.plot(range(nb_epochs), train_loss_t, color='orange', label='Loss on the training set')
    plt.plot(range(nb_epochs), test_loss_t, color='green', label='Loss on the testing set')
    plt.legend()
    plt.show()

In [ ]:
plot_losses(train_loss_t, test_loss_t)

## 5. 查看你的结果

为了计算基线，我计算训练数据集中所有故障时间的平均值。


In [ ]:
max_val = np.zeros(len(y_train))
for i,y in enumerate(y_train):
    max_val[i] = y[0].item()
baseline = np.mean(max_val)

这里我计算我的模型在测试集上做出的所有预测，并导出到 `numpy`。


In [ ]:
def compute_np(model,X_test, y_test,baseline=baseline,device=device,max_size=303):
    n_test = len(X_test)
    all_pred = np.empty((n_test,max_size,2))
    all_y = np.empty((n_test,max_size))
    base_pred = np.empty((n_test,max_size))
    all_pred[:] = np.NaN
    all_y[:] = np.NaN
    base_pred[:] = np.NaN
    list_npred = []
    for k in range(n_test):
        pred = model(X_test[k].to(device))
        pred_np = pred.cpu().detach().numpy()
        n_pred = pred_np.shape[0]
        list_npred.append(n_pred)
        all_pred[k,:n_pred,:] = pred_np
        all_y[k,:n_pred] = y_test[k].numpy()
        base_pred[k,:n_pred] = baseline - range(n_pred)
    return all_pred, all_y, base_pred, list_npred

In [ ]:
all_pred, all_y, base_pred, list_npred = compute_np(model, X_test,y_test)
pred_fail = all_pred[:,:,0]*gamma(1+1/all_pred[:,:,1])

在测试集上，我们只能访问序列的开头，需要预测故障时间。对给定的发动机，你可以比较模型做出的预测、基线和真实值：


In [ ]:
k = 50
plt.plot(pred_fail[k,:],label='predicted')
plt.plot(all_y[k],label='true')
plt.plot(base_pred[k],label='baseline')
plt.legend()

为了得到性能度量，我们计算 RMSE：


In [ ]:
def RMSE(pred_fail, all_y):
    return np.sqrt((pred_fail-all_y)**2)

In [ ]:
res= RMSE(pred_fail,all_y)
res_base = RMSE(base_pred,all_y)

RMSE 误差就是上面的估计线与真实线之间的距离。对基线它是常数，而随着我们的模型获得越来越多的数据，它应该下降。下面是上面那个具体例子的演示：


In [ ]:
plt.plot(res[k])
plt.plot(res_base[k])

下面，我对整个数据集上的 RMSE 取平均，保留时间轴（注意每个点都是平均，但样本数并不相同）。

可以看到基线的 RMSE 对长序列非常差。这应该在意料之中，因为这些长序列对应的是健康的发动机！

相反，我们的模型随着输入序列长度的增加，RMSE 在下降。

![](https://raw.githubusercontent.com/mlelarge/dataflowr/master/PlutonAI/img/rmse.png)


In [ ]:
plt.plot(np.nanmean(res,0), label = 'RMSE')
plt.plot(np.nanmean(res_base,0), label ='RMSE baseline')
plt.legend()

In [ ]:
np.nanmean(res)

In [ ]:
np.nanmean(res_base)

In [ ]:
last_indices = list((~np.isnan(res)).sum(axis = 1) - 1)

In [ ]:
np.mean([res[i,j] for i,j in enumerate(last_indices)])

In [ ]:
np.mean([res_base[i,j] for i,j in enumerate(last_indices)])

上面可以看到，与基线相比，我们把 RMSE 降低了超过一半。

这里我们画基线的预测值 vs 真实值的散点图（越靠近蓝色对角线越好）：

![](https://raw.githubusercontent.com/mlelarge/dataflowr/master/PlutonAI/img/base_scatter.png)


In [ ]:
plot = sns.jointplot(x=[all_y[i,j] for i,j in enumerate(last_indices)],y=[base_pred[i,j] for i,j in enumerate(last_indices)],dropna=True,kind="kde", n_levels=30, color="g");
plot.ax_joint.plot([0,150], [0,150], 'b-', linewidth = 2);
plot.set_axis_labels('true', 'predicted');

现在用我们的模型做同样的散点图（越靠近蓝色对角线越好）。可以看到巨大的改进。

![](https://raw.githubusercontent.com/mlelarge/dataflowr/master/PlutonAI/img/model_scatter.png)


In [ ]:
plot = sns.jointplot(x=[all_y[i,j] for i,j in enumerate(last_indices)],y=[pred_fail[i,j] for i,j in enumerate(last_indices)],dropna=True,kind="kde", n_levels=30, color="g");
plot.ax_joint.plot([0,150], [0,150], 'b-', linewidth = 2);
plot.set_axis_labels('true', 'predicted');

In [ ]:
plot = sns.jointplot(x=all_y,y=pred_fail,dropna=True,kind="kde", space=0, color="g");
plot.ax_joint.plot([0,200], [0,200], 'b-', linewidth = 2);
plot.set_axis_labels('true', 'predicted');

[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)